# RMSProp

## Definition: RMSProp (Root Mean Square Propagation)

RMSProp (Hinton, 2012) adapts the learning rate per parameter using an
exponential moving average of squared gradients, essentially a heuristic,
exponentially weighted version of AdaGrad. Unlike AdaGrad, it is not derived
from a convex optimization objective; it was introduced empirically to
stabilize deep-network training (especially RNNs) by adapting step sizes
without letting them decay to zero. The running average of squared gradients
serves as a measure of how “safe” each direction is: directions with large or
unstable slopes receive smaller steps to avoid overshooting, while flatter
directions receive larger ones.

Because this rescaling partially normalizes each gradient coordinate, RMSProp
often moves in a direction that resembles the sign pattern of the gradient more
than its exact magnitudes. In high-dimensional settings this is not a flaw,
since the sign, that is the octant of the gradient, captures most of the useful
directional information. This type of octant selection can be surprisingly
effective, and sometimes even preferable, under noisy gradients.

Given the gradient at step $t$:
$$
g_t = \nabla_\theta L(\theta_t),
$$

RMSProp keeps an exponential moving average of the component-wise squared
gradients:
$$
s_{t+1} = \alpha\, s_t + (1 - \alpha)\, g_t^2, \qquad s_0 = 0_{\mathbb{R}^{d_\theta}},
$$
where $d_\theta$ denotes the parameter dimension and $\alpha \in [0,1)$ is
the decay rate (common values: $0.9$ or $0.99$).

The parameter update is:
$$
\theta_{t+1} = \theta_t - \eta\, \frac{g_t}{\sqrt{s_{t+1}} + \varepsilon},
$$

where  
• $\eta$ is the learning rate,  
• $\varepsilon > 0$ is a small constant used for numerical stability.

### Property: Exponential Forgetting (Geometric Weighting)
$$
s_{n+1} = (1-\alpha)\sum_{k=0}^{n}
      \alpha^{\,n-k} g_k^2.
$$

Thus, the contribution of a past gradient $g_k^2$ to $s_t$ is
$$
(1-\alpha)\,\alpha^{\,t-1-k},
$$
which decays geometrically with age.

**Proof:** Let's prove it by induction on $n$. 

For $n=1$, we have $s_{1}=(1-\alpha)g^2_0$ on the left side, and on the right side we have 
$$(1-\alpha)\sum_{k=0}^0\alpha^{1-1-k}g^2_{k}=(1-\alpha)\alpha^{0}g^2_{0}=(1-\alpha)g^2_{0},$$
so the equality is true for $n=1$.

Let's prove the induction step, assume the equation is true for $n$, then we have

$$
\begin{align*}
s_{n+1} &= \alpha s_{n} + (1-\alpha)g^2_{n},\\
&=\alpha (1-\alpha)\sum_{k=0}^{n-1}
      \alpha^{n-1-k} g_k^2+ (1-\alpha)g^2_{n},\\
&=(1-\alpha)\sum_{k=0}^{n-1}
      \alpha^{n-k} g_k^2+ (1-\alpha)g^2_{n},\\
&=(1-\alpha)\left(\sum_{k=0}^{n-1}
      \alpha^{n-k} g_k^2+g^2_{n}\right),\\
&=(1-\alpha)\sum_{k=0}^{n}
      \alpha^{n-k} g_k^2.
\end{align*}
$$
so it is also true for $n+1$ and the inducion is completed.

## Code: RMSProp

In [ ]:
import torch
from matplotlib import pyplot as plt
from torch import Tensor, nn

import optimizers as mynn
from models.deep_learning.architectures import MLP

### Data

In [ ]:
def f(x: Tensor) -> Tensor:
    return 1 + 2 * x**2


N = 20
xs = (4 * torch.rand(N) - 2).unsqueeze(1)
ys = f(xs) + 0.5 * torch.randn(N).unsqueeze(1)

x = torch.linspace(-2, 2, 100, requires_grad=True)
x_eval = torch.linspace(-2, 2, 100)
torch.manual_seed(1)

In [ ]:
plt.plot(xs.detach().numpy(), ys.detach().numpy(), "o", label="data")
plt.plot(x.detach().numpy(), f(x).detach().numpy(), "--", label="true")
plt.legend()
plt.grid()
plt.show()


### Training loop

In [ ]:
def train(
    model: nn.Module,
    opt: torch.optim.Optimizer,
    loss_fn: nn.Module,
    data: tuple[Tensor, Tensor],
    epochs: int = 20,
    batch_size: int = 1,
):
    xs, ys = data
    N = xs.shape[0]
    model.train()
    losses: list[float] = []
    for epoch in range(epochs):
        perm = torch.randperm(N)  # shuffle for stochasticity each epoch
        loss = torch.tensor(torch.inf)
        for i in range(0, N, batch_size):
            batch_idx = perm[i : i + batch_size]
            x = xs[batch_idx]
            y = ys[batch_idx]
            opt.zero_grad()
            pred = model(x)
            loss = loss_fn(pred, y)
            loss.backward()
            opt.step()
        losses.append(loss.item())
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss.item():.4f}")
    return losses


def compare_losses(loss: list[float], nn_loss: list[float], title: str):
    plt.plot(nn_loss, linestyle="-", label="nn_loss")
    plt.plot(loss, linestyle="--", label="loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.show()

## Parameters

In [ ]:
lr = 0.001
momentum = 0.9
dampening = 0.0
weight_decay = 0.001
nesterov = False

loss_fn = nn.MSELoss()

### pytorch

In [ ]:
torch.manual_seed(1)
model = MLP(
    input_dim=1, output_dim=1, hidden_dims=[512, 512, 512], activation_cls=nn.Tanh
)
opt = torch.optim.RMSprop(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay,
)

nn_loss = train(model, opt, loss_fn, (xs, ys), epochs=150, batch_size=N)

### Custom

In [ ]:
torch.manual_seed(1)
model = MLP(
    input_dim=1, output_dim=1, hidden_dims=[512, 512, 512], activation_cls=nn.Tanh
)
opt = mynn.RMSProp(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay,
)

loss = train(model, opt, loss_fn, (xs, ys), epochs=150, batch_size=N)
compare_losses(loss, nn_loss, "RMSProp (Full-Batch) Loss")

# Evaluation

In [ ]:
plt.plot(
    x_eval, model(x_eval[:, None]).squeeze().detach().numpy(), "-", label="learned"
)
plt.plot(xs.detach().numpy(), ys.detach().numpy(), "o", label="data")
plt.plot(x_eval, f(x_eval), "--", label="true")
plt.title("RMSProp Learned Function")
plt.legend()
plt.grid()
plt.show()